In [1]:
"""
Causal SHAP for Soil SPI dataset — known causal graph + ground-truth gamma_i.

Key design choices vs the PC-based pipeline:
  - Causal graph is the observed-feature DAG from the DGP: OC -> pH -> Zinc,
    and all 7 features -> SPI. The full DGP also contains a latent quality
    variable (quality -> OC, N, P, K, ZN_base) not representable in the
    observed feature set; its shared dependence is noted but not modelled.
  - gamma_i are derived from ground_truth.csv (exact per-observation net
    contributions) rather than analytically, because this DGP uses mixed
    functional forms (log, sqrt, exp, asymmetric penalties) with high
    state-dependence (CV 40-105%). The mean operating-point approximation
    valid for the translog yield model is NOT valid here.
  - PC discovery, IDA DAG sampling, W_i regression are all removed.

gamma_i derivation is documented in soil_gamma_derivation.md.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
import networkx as nx

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# Config
CONFIG = {
    'DATA_PATH':        '../datasets/soil_spi_preprocessed.npz',
    'GROUND_TRUTH_PATH':'../datasets/ground_truth.csv',   # exact per-obs contributions
    'MAX_PERM':    200,
    'M_RESAMPLES': 64,
    'GLOBAL_K':    30,
    'RANDOM_SEED': 42,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

# Feature names
#  NPZ uses long names; DGP columns use short names.
FEATURE_NAMES = [           # long names (NPZ)
    'Organic Carbon',
    'Nitrogen',
    'Phosphorus',
    'Potassium',
    'pH',
    'Zinc',
    'Electrical Conductivity',
]

SHORT_NAMES = ['OC', 'N', 'P', 'K', 'PH', 'ZN', 'EC']   # DGP column names

CAUSAL_GRAPH: List[Tuple[str, str]] = [
    # Inter-feature structural edges (from DGP)
    ('Organic Carbon', 'pH'),          # OC shifts PH downward
    ('pH',             'Zinc'),        # high PH suppresses ZN bioavailability
    # Direct edges to target
    ('Organic Carbon',           'SPI'),
    ('Nitrogen',                 'SPI'),
    ('Phosphorus',               'SPI'),
    ('Potassium',                'SPI'),
    ('pH',                       'SPI'),
    ('Zinc',                     'SPI'),
    ('Electrical Conductivity',  'SPI'),
]

#  GAMMA COMPUTATION  (from exact ground-truth contributions)
#  Because the SPI DGP uses mixed functional forms with high
#  state-dependence (CV 40-105%), the mean operating-point
#  approximation used in the translog yield model is not valid.
#  Instead, gamma_i = E[|net_i|] / sum_j E[|net_j|] is computed
#  directly from the exact per-observation contributions in
#  ground_truth.csv, which constitutes a stronger justification.
def compute_gamma(ground_truth_path: str, verbose: bool = True) -> Dict[str, float]:
    """
    Derive gamma_i from the exact per-observation ground-truth contributions.

    Net contribution per feature per observation:
        net_OC = OC_base - OC_penalty
        net_N  = N_base  - N_penalty
        net_P  = P_base  - P_penalty
        net_K  = K_base  - K_penalty
        net_PH = -PH_penalty          (PH enters only as a penalty)
        net_ZN = ZN_base - ZN_penalty
        net_EC = -EC_penalty          (EC enters only as a penalty)

    gamma_i = E[|net_i|] / sum_j E[|net_j|]

    This is computed over the full dataset (train + test combined),
    which equals the DGP population used to produce ground_truth.csv.
    """
    gt = pd.read_csv(ground_truth_path)

    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })

    mean_abs = net.abs().mean()           # E[|net_i|] over full dataset
    cv       = net.std() / net.abs().mean()  # coefficient of variation
    total    = mean_abs.sum()
    gamma    = (mean_abs / total).to_dict()

    if verbose:
        print("\n" + "=" * 68)
        print("gamma_i derivation from exact ground-truth contributions")
        print("=" * 68)
        print(f"  {'Feature':<28} {'E[|net|]':>10} {'CV(%)':>8} {'gamma_i':>10}")
        print("  " + "-" * 58)
        for f in FEATURE_NAMES:
            print(f"  {f:<28} {mean_abs[f]:>10.4f} {100*cv[f]:>7.1f}% {gamma[f]:>10.6f}")
        print("  " + "-" * 58)
        print(f"  {'Total':<28} {total:>10.4f} {'':>8} {sum(gamma.values()):>10.4f}")
        print("=" * 68)
        print("  NOTE: CV 40-105% — mean operating-point approximation is NOT")
        print("  valid for this DGP. Ground-truth gamma_i is the correct choice.\n")

    return gamma


#  BUILD FEATURE-ONLY DAG  (SPI node excluded; used by SEM)
def build_known_dag() -> nx.DiGraph:
    """
    Build the observed-feature DAG.
    Encodes OC -> pH and pH -> Zinc (structural dependencies present in the
    observed feature set). The latent quality variable (quality -> OC, N, P,
    K, ZN_base) is not representable here and is noted but not modelled.
    """
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in CAUSAL_GRAPH:
        if v != 'SPI':
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G), "CAUSAL_GRAPH contains a cycle!"
    edges = [(u, v) for u, v in CAUSAL_GRAPH if v != 'SPI']
    to_spi = [u for u, v in CAUSAL_GRAPH if v == 'SPI']
    print(f"Observed-feature DAG: {G.number_of_nodes()} nodes, "
          f"inter-feature edges {edges}.")
    print(f"  Direct edges to SPI: {to_spi}")
    print("  Note: latent quality variable (-> OC, N, P, K, ZN_base) "
          "is not represented in the observed DAG.")
    return G


#  LINEAR-GAUSSIAN SEM  (unchanged from yield pipeline)
class LinearMechanisms:
    """Linear-Gaussian SEM on the known DAG for interventional sampling."""

    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag           = dag
        self.feature_names = feature_names
        self.mechanisms: Dict[str, dict] = {}
        self.topo: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        self.topo = list(nx.topological_sort(self.dag))
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        """One draw from P(X_{not S} | do(X_S = x_S)) via the fitted SEM."""
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            mech  = self.mechanisms[var]
            sigma = mech['sigma']
            if not mech['parents']:
                mu = mech['intercept']
            else:
                pvals = np.array([x_new.get(p, float(x_obs[p]))
                                  for p in mech['parents']])
                mu = mech['intercept'] + pvals @ mech['coef']
            x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new


#  CAUSAL SHAP  (paper Eqs. 4, 10, 11)
class CausalSHAP:
    """
    Causal SHAP with known DAG and ground-truth-derived gamma_i.

    vc(S)  (Eq. 4): E[f(X) | do(X_S = x_S)] via MC from the fitted SEM.
    phi_i  (Eq. 10): Shapley MC with marginal contributions weighted by gamma_i.
    Normalized (Eq. 11): sum(phi_i) = f(x) - E[f(X)].
    """

    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float],
                 X_train: pd.DataFrame, y_train: pd.Series):
        self.model    = model
        self.dag      = dag
        self.gamma    = gamma
        self.X_train  = X_train
        self.features = list(X_train.columns)
        self._Ef_cache = None

        print("\nFitting linear-Gaussian SEM on known DAG ...")
        self.sem = LinearMechanisms(dag, self.features).fit(X_train)
        print("SEM fitted. Ready.\n")
        print("gamma_i (causal weights):")
        for f in self.features:
            print(f"  {f:<30} gamma = {self.gamma[f]:.6f}")

    def _Ef(self, cls: int) -> float:
        if self._Ef_cache is None:
            self._Ef_cache = self.model.predict_proba(
                self.X_train.values).mean(axis=0)
        return float(self._Ef_cache[cls])

    def _vc(self, S: Set[str], x: pd.Series, cls: int, M: int) -> float:
        total = 0.0
        for _ in range(M):
            x_s   = self.sem.sample_intervention(S, x)
            X_mat = pd.DataFrame([x_s], columns=self.features).values
            total += self.model.predict_proba(X_mat)[0][cls]
        return total / M

    def explain_instance(self, x: pd.Series,
                         n_perms: int = None, M: int = None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M       is None: M       = CONFIG['M_RESAMPLES']

        probs = self.model.predict_proba(
            pd.DataFrame([x], columns=self.features).values)[0]
        cls = int(np.argmax(probs))
        fx  = float(probs[cls])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, cls, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, cls, M)
                phi[self.features.index(feat)] += \
                    self.gamma[feat] * (v_curr - v_prev)    # Eq. 10
                v_prev, S = v_curr, S_next
        phi /= n_perms

        # Eq. 11 normalization
        sum_phi = float(np.sum(phi))
        if abs(sum_phi) > 1e-12:
            phi = phi * ((fx - self._Ef(cls)) / sum_phi)

        return pd.Series(phi, index=self.features, name='Causal_SHAP')

    def explain_global(self, X_test: pd.DataFrame, k: int = None) -> pd.Series:
        if k is None: k = CONFIG['GLOBAL_K']
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nGlobal Causal SHAP over {k} test instances ...")
        results = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            results.append(self.explain_instance(
                X_test.iloc[i],
                n_perms=max(10, CONFIG['MAX_PERM'] // 4),
            ))
        return pd.concat(results, axis=1).mean(axis=1).rename('Causal_SHAP_Global')


# Baselines
def compute_kernel_shap(model, X_train, X_test):
    try:
        import shap
        print("\nComputing Kernel SHAP (baseline) ...")
        bg  = shap.kmeans(X_train.values, min(100, len(X_train)))
        exp = shap.KernelExplainer(lambda x: model.predict_proba(x), bg)
        sv  = exp.shap_values(X_test.values, nsamples=100)
        glob = (np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
                if isinstance(sv, list) else np.abs(sv).mean(axis=0))
        return pd.Series(glob, index=X_test.columns, name='Kernel_SHAP')
    except Exception as e:
        print(f"Kernel SHAP skipped: {e}"); return None


def model_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names,
                         name='Model_Importance')
    return None


# Plots
def plot_causal_graph(dag: nx.DiGraph, gamma: Dict[str, float]):
    G_full = dag.copy()
    G_full.add_node('SPI')
    for (u, v) in CAUSAL_GRAPH:
        if v == 'SPI':
            G_full.add_edge(u, 'SPI')

    fig, ax = plt.subplots(figsize=(12, 6))
    # Hand-placed layout for clarity
    pos = {
        'Organic Carbon':           (0.05, 0.80),
        'Nitrogen':                 (0.22, 0.80),
        'Phosphorus':               (0.38, 0.80),
        'Potassium':                (0.54, 0.80),
        'pH':                       (0.14, 0.45),
        'Zinc':                     (0.30, 0.45),
        'Electrical Conductivity':  (0.78, 0.80),
        'SPI':                      (0.44, 0.08),
    }

    feat_nodes = [n for n in G_full.nodes() if n != 'SPI']
    nx.draw_networkx_nodes(G_full, pos,
                           nodelist=feat_nodes,
                           node_color='#5DCAA5', node_size=1600, alpha=0.9, ax=ax)
    nx.draw_networkx_nodes(G_full, pos,
                           nodelist=['SPI'],
                           node_color='#7F77DD', node_size=2000, alpha=0.9, ax=ax)

    short_labels = {
        'Organic Carbon': 'OC', 'Nitrogen': 'N', 'Phosphorus': 'P',
        'Potassium': 'K', 'pH': 'pH', 'Zinc': 'ZN',
        'Electrical Conductivity': 'EC', 'SPI': 'SPI',
    }
    nx.draw_networkx_labels(G_full, pos, labels=short_labels,
                            font_size=9, font_weight='bold', ax=ax)

    # Inter-feature edges (structural)
    inter_edges = [(u, v) for u, v in CAUSAL_GRAPH if v != 'SPI']
    nx.draw_networkx_edges(G_full, pos, edgelist=inter_edges,
                           edge_color='#E08020', arrows=True, arrowsize=18,
                           width=2.0, connectionstyle='arc3,rad=0.15', ax=ax)

    # Edges to SPI
    spi_edges = [(u, v) for u, v in CAUSAL_GRAPH if v == 'SPI']
    nx.draw_networkx_edges(G_full, pos, edgelist=spi_edges,
                           edge_color='#3B6EA5', arrows=True, arrowsize=18,
                           width=2.0, connectionstyle='arc3,rad=0.05', ax=ax)

    # Annotate gamma_i on SPI edges
    for (u, v) in spi_edges:
        if u in gamma:
            x0, y0 = pos[u]; x1, y1 = pos['SPI']
            ax.text((x0+x1)/2 + 0.03, (y0+y1)/2,
                    f'g={gamma[u]:.3f}',
                    fontsize=7, color='#0C447C', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white',
                              alpha=0.85, ec='#B5D4F4', lw=0.7))

    from matplotlib.lines import Line2D
    legend = [
        Line2D([0],[0], color='#E08020', lw=2, label='Structural (OC→pH→ZN)'),
        Line2D([0],[0], color='#3B6EA5', lw=2, label='Direct effect on SPI'),
    ]
    ax.legend(handles=legend, loc='lower right', fontsize=8)
    ax.set_title(
        'Observed-feature causal graph (Soil SPI DGP)\n'
        'Latent quality variable not shown  |  g = gamma_i from ground_truth.csv',
        fontsize=10, fontweight='bold'
    )
    ax.axis('off'); fig.tight_layout()
    return fig


def plot_attributions(df: pd.DataFrame):
    n_cols = df.shape[1]
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 6), sharey=False)
    if n_cols == 1: axes = [axes]
    for ax, col in zip(axes, df.columns):
        vals   = df[col].values
        colors = ['#3B8BD4' if v >= 0 else '#E24B4A' for v in vals]
        bars   = ax.barh(df.index.tolist(), vals, color=colors,
                         edgecolor='white', height=0.55)
        ax.axvline(0, color='#888', linewidth=0.8, linestyle='--')
        for bar, val in zip(bars, vals):
            ax.text(val + (0.0005 if val >= 0 else -0.0005),
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center',
                    ha='left' if val >= 0 else 'right', fontsize=8)
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel('Attribution score')
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(
        'Feature Attribution — Soil SPI (known causal graph, ground-truth gamma_i)',
        fontsize=11, fontweight='bold', y=1.01
    )
    fig.tight_layout()
    return fig


# Main
def main():
    print("\n" + "="*70)
    print("CAUSAL SHAP — Soil SPI (known causal graph, ground-truth gamma_i)")
    print("="*70 + "\n")

    # Load NPZ
    try:
        data = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X_train = pd.DataFrame(data['X_train'], columns=list(data['feature_names']))
    X_test  = pd.DataFrame(data['X_test'],  columns=list(data['feature_names']))
    y_train = pd.Series(data['y_train'].astype(int))
    y_test  = pd.Series(data['y_test'].astype(int))
    print(f"Loaded: X_train={X_train.shape}, X_test={X_test.shape}")
    print(f"Features: {list(X_train.columns)}")

    # gamma_i from exact ground truth (full dataset, unscaled)
    try:
        gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'], verbose=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['GROUND_TRUTH_PATH']}' not found. "
              "Place ground_truth.csv in the working directory."); return

    # Scale for model training and SEM fitting
    # gamma is derived from unscaled contributions - scaling only affects SEM internals
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    Xte = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    # Train model
    n_classes = int(len(np.unique(y_train)))
    print(f"Training model ({n_classes} classes: Low/Medium/High SPI) ...")
    if HAVE_XGB:
        model = XGBClassifier(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
            objective='multi:softprob', num_class=n_classes,
        )
    else:
        model = RandomForestClassifier(
            n_estimators=160, max_depth=6,
            random_state=CONFIG['RANDOM_SEED'])
    model.fit(Xtr.values, y_train.values)
    y_pred = model.predict(Xte.values)
    print(f"Accuracy={accuracy_score(y_test, y_pred):.4f}  "
          f"MacroF1={f1_score(y_test, y_pred, average='macro'):.4f}")

    # Build DAG and save graph
    dag   = build_known_dag()
    fig_g = plot_causal_graph(dag, gamma)
    fig_g.savefig('causal_graph_soil.png', dpi=150, bbox_inches='tight')
    print("Saved causal_graph_soil.png")

    # Causal SHAP
    cshap      = CausalSHAP(model=model, dag=dag, gamma=gamma,
                             X_train=Xtr, y_train=y_train)
    global_phi = cshap.explain_global(Xte, k=CONFIG['GLOBAL_K'])
    print("\nGlobal Causal SHAP (sorted by |attribution|):")
    print(global_phi.sort_values(key=abs, ascending=False).to_string())

    # Baselines
    kshap = compute_kernel_shap(model, Xtr, Xte)
    mimp  = model_importance(model, list(Xtr.columns))

    # Save
    cols = {'Causal_SHAP': global_phi}
    if kshap is not None: cols['Kernel_SHAP']     = kshap
    if mimp  is not None: cols['Model_Importance'] = mimp
    df = pd.DataFrame(cols)
    df.to_csv('attribution_scores_soil.csv')
    print("\n" + df.round(6).to_string())
    print("\nSaved attribution_scores_soil.csv")

    fig_a = plot_attributions(df)
    fig_a.savefig('attribution_comparison_soil.png', dpi=150, bbox_inches='tight')
    print("Saved attribution_comparison_soil.png")


if __name__ == '__main__':
    main()


CAUSAL SHAP — Soil SPI (known causal graph, ground-truth gamma_i)

Loaded: X_train=(4000, 7), X_test=(1000, 7)
Features: [np.str_('Organic Carbon'), np.str_('Nitrogen'), np.str_('Phosphorus'), np.str_('Potassium'), np.str_('pH'), np.str_('Zinc'), np.str_('Electrical Conductivity')]

gamma_i derivation from exact ground-truth contributions
  Feature                        E[|net|]    CV(%)    gamma_i
  ----------------------------------------------------------
  Organic Carbon                   6.5455    84.5%   0.062171
  Nitrogen                        42.2350    55.2%   0.401157
  Phosphorus                       8.0212    41.8%   0.076187
  Potassium                       27.2443    51.0%   0.258773
  pH                               9.3296   104.8%   0.088614
  Zinc                             1.5475    91.9%   0.014698
  Electrical Conductivity         10.3598    66.6%   0.098400
  ----------------------------------------------------------
  Total                          105.282

Kernel SHAP skipped: Data must be 1-dimensional, got ndarray of shape (7, 3) instead

                         Causal_SHAP  Model_Importance
Organic Carbon              0.012490          0.114053
Nitrogen                    0.146858          0.250705
Phosphorus                 -0.003150          0.066715
Potassium                   0.333133          0.251575
pH                          0.038116          0.186356
Zinc                       -0.000448          0.034015
Electrical Conductivity     0.035450          0.096580

Saved attribution_scores_soil.csv
Saved attribution_comparison_soil.png


In [3]:
"""
Causal SHAP for Soil SPI dataset — known causal graph + ground-truth gamma_i.

Key design choices vs the PC-based pipeline:
  - Causal graph is the observed-feature DAG from the DGP: OC -> pH -> Zinc,
    and all 7 features -> SPI. The full DGP also contains a latent quality
    variable (quality -> OC, N, P, K, ZN_base) not representable in the
    observed feature set; its shared dependence is noted but not modelled.
  - gamma_i are derived from ground_truth.csv (exact per-observation net
    contributions) rather than analytically, because this DGP uses mixed
    functional forms (log, sqrt, exp, asymmetric penalties) with high
    state-dependence (CV 40-105%). The mean operating-point approximation
    valid for the translog yield model is NOT valid here.
  - PC discovery, IDA DAG sampling, W_i regression are all removed.

gamma_i derivation is documented in soil_gamma_derivation.md.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
import networkx as nx

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# Config
CONFIG = {
    'DATA_PATH':        '../datasets/soil_spi_preprocessed.npz',
    'GROUND_TRUTH_PATH':'../datasets/ground_truth.csv',   # exact per-obs contributions
    'MAX_PERM':    200,
    'M_RESAMPLES': 64,
    'GLOBAL_K':    30,
    'RANDOM_SEED': 42,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

# Feature names
#  NPZ uses long names; DGP columns use short names.
FEATURE_NAMES = [           # long names (NPZ)
    'Organic Carbon',
    'Nitrogen',
    'Phosphorus',
    'Potassium',
    'pH',
    'Zinc',
    'Electrical Conductivity',
]

SHORT_NAMES = ['OC', 'N', 'P', 'K', 'PH', 'ZN', 'EC']   # DGP column names

CAUSAL_GRAPH: List[Tuple[str, str]] = [
    # Inter-feature structural edges (from DGP)
    ('Organic Carbon', 'pH'),          # OC shifts PH downward
    ('pH',             'Zinc'),        # high PH suppresses ZN bioavailability
    # Direct edges to target
    ('Organic Carbon',           'SPI'),
    ('Nitrogen',                 'SPI'),
    ('Phosphorus',               'SPI'),
    ('Potassium',                'SPI'),
    ('pH',                       'SPI'),
    ('Zinc',                     'SPI'),
    ('Electrical Conductivity',  'SPI'),
]

#  GAMMA COMPUTATION  (from exact ground-truth contributions)
#  Because the SPI DGP uses mixed functional forms with high
#  state-dependence (CV 40-105%), the mean operating-point
#  approximation used in the translog yield model is not valid.
#  Instead, gamma_i = E[|net_i|] / sum_j E[|net_j|] is computed
#  directly from the exact per-observation contributions in
#  ground_truth.csv, which constitutes a stronger justification.
def compute_gamma(ground_truth_path: str, verbose: bool = True) -> Dict[str, float]:
    """
    Derive gamma_i from the exact per-observation ground-truth contributions.

    Net contribution per feature per observation:
        net_OC = OC_base - OC_penalty
        net_N  = N_base  - N_penalty
        net_P  = P_base  - P_penalty
        net_K  = K_base  - K_penalty
        net_PH = -PH_penalty          (PH enters only as a penalty)
        net_ZN = ZN_base - ZN_penalty
        net_EC = -EC_penalty          (EC enters only as a penalty)

    gamma_i = E[|net_i|] / sum_j E[|net_j|]

    This is computed over the full dataset (train + test combined),
    which equals the DGP population used to produce ground_truth.csv.
    """
    gt = pd.read_csv(ground_truth_path)

    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })

    mean_abs = net.abs().mean()           # E[|net_i|] over full dataset
    cv       = net.std() / net.abs().mean()  # coefficient of variation
    total    = mean_abs.sum()
    gamma    = (mean_abs / total).to_dict()

    if verbose:
        print("\n" + "=" * 68)
        print("gamma_i derivation from exact ground-truth contributions")
        print("=" * 68)
        print(f"  {'Feature':<28} {'E[|net|]':>10} {'CV(%)':>8} {'gamma_i':>10}")
        print("  " + "-" * 58)
        for f in FEATURE_NAMES:
            print(f"  {f:<28} {mean_abs[f]:>10.4f} {100*cv[f]:>7.1f}% {gamma[f]:>10.6f}")
        print("  " + "-" * 58)
        print(f"  {'Total':<28} {total:>10.4f} {'':>8} {sum(gamma.values()):>10.4f}")
        print("=" * 68)
        print("  NOTE: CV 40-105% — mean operating-point approximation is NOT")
        print("  valid for this DGP. Ground-truth gamma_i is the correct choice.\n")

    return gamma


#  BUILD FEATURE-ONLY DAG  (SPI node excluded; used by SEM)
def build_known_dag() -> nx.DiGraph:
    """
    Build the observed-feature DAG.
    Encodes OC -> pH and pH -> Zinc (structural dependencies present in the
    observed feature set). The latent quality variable (quality -> OC, N, P,
    K, ZN_base) is not representable here and is noted but not modelled.
    """
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in CAUSAL_GRAPH:
        if v != 'SPI':
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G), "CAUSAL_GRAPH contains a cycle!"
    edges = [(u, v) for u, v in CAUSAL_GRAPH if v != 'SPI']
    to_spi = [u for u, v in CAUSAL_GRAPH if v == 'SPI']
    print(f"Observed-feature DAG: {G.number_of_nodes()} nodes, "
          f"inter-feature edges {edges}.")
    print(f"  Direct edges to SPI: {to_spi}")
    print("  Note: latent quality variable (-> OC, N, P, K, ZN_base) "
          "is not represented in the observed DAG.")
    return G


#  LINEAR-GAUSSIAN SEM  (unchanged from yield pipeline)
class LinearMechanisms:
    """Linear-Gaussian SEM on the known DAG for interventional sampling."""

    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag           = dag
        self.feature_names = feature_names
        self.mechanisms: Dict[str, dict] = {}
        self.topo: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        self.topo = list(nx.topological_sort(self.dag))
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        """One draw from P(X_{not S} | do(X_S = x_S)) via the fitted SEM."""
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            mech  = self.mechanisms[var]
            sigma = mech['sigma']
            if not mech['parents']:
                mu = mech['intercept']
            else:
                pvals = np.array([x_new.get(p, float(x_obs[p]))
                                  for p in mech['parents']])
                mu = mech['intercept'] + pvals @ mech['coef']
            x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new


#  CAUSAL SHAP  (paper Eqs. 4, 10, 11)
class CausalSHAP:
    """
    Causal SHAP with known DAG and ground-truth-derived gamma_i.

    vc(S)  (Eq. 4): E[f(X) | do(X_S = x_S)] via MC from the fitted SEM.
    phi_i  (Eq. 10): Shapley MC with marginal contributions weighted by gamma_i.
    Normalized (Eq. 11): sum(phi_i) = f(x) - E[f(X)].
    """

    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float],
                 X_train: pd.DataFrame, y_train: pd.Series):
        self.model    = model
        self.dag      = dag
        self.gamma    = gamma
        self.X_train  = X_train
        self.features = list(X_train.columns)
        self._Ef_cache = None

        print("\nFitting linear-Gaussian SEM on known DAG ...")
        self.sem = LinearMechanisms(dag, self.features).fit(X_train)
        print("SEM fitted. Ready.\n")
        print("gamma_i (causal weights):")
        for f in self.features:
            print(f"  {f:<30} gamma = {self.gamma[f]:.6f}")

    def _Ef(self, cls: int) -> float:
        if self._Ef_cache is None:
            self._Ef_cache = self.model.predict_proba(
                self.X_train.values).mean(axis=0)
        return float(self._Ef_cache[cls])

    def _vc(self, S: Set[str], x: pd.Series, cls: int, M: int) -> float:
        total = 0.0
        for _ in range(M):
            x_s   = self.sem.sample_intervention(S, x)
            X_mat = pd.DataFrame([x_s], columns=self.features).values
            total += self.model.predict_proba(X_mat)[0][cls]
        return total / M

    def explain_instance(self, x: pd.Series,
                         n_perms: int = None, M: int = None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M       is None: M       = CONFIG['M_RESAMPLES']

        probs = self.model.predict_proba(
            pd.DataFrame([x], columns=self.features).values)[0]
        cls = int(np.argmax(probs))
        fx  = float(probs[cls])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, cls, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, cls, M)
                phi[self.features.index(feat)] += \
                    self.gamma[feat] * (v_curr - v_prev)    # Eq. 10
                v_prev, S = v_curr, S_next
        phi /= n_perms

        # Eq. 11 normalization
        sum_phi = float(np.sum(phi))
        if abs(sum_phi) > 1e-12:
            phi = phi * ((fx - self._Ef(cls)) / sum_phi)

        return pd.Series(phi, index=self.features, name='Causal_SHAP')

    def explain_global(self, X_test: pd.DataFrame, k: int = None):
        """
        Returns three objects:
          global_signed  — mean(phi_i)       per feature: direction + cancellation
          global_abs     — mean(|phi_i|)      per feature: magnitude / importance
          phi_mat        — full (features x k) local attribution matrix

        Signed mean and absolute mean are both needed because gamma_i is
        magnitude-based (gamma_i = E[|net_i|] / sum). Comparing gamma_i against
        signed mean attribution conflates direction with importance. The fair
        global importance comparison is mean absolute attribution.
        """
        if k is None:
            k = CONFIG['GLOBAL_K']
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nGlobal Causal SHAP over {k} test instances ...")
        results = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            results.append(
                self.explain_instance(
                    X_test.iloc[i],
                    n_perms=max(10, CONFIG['MAX_PERM'] // 4),
                )
            )
        phi_mat       = pd.concat(results, axis=1)
        global_signed = phi_mat.mean(axis=1).rename('Causal_SHAP_Signed')
        global_abs    = phi_mat.abs().mean(axis=1).rename('Causal_SHAP_AbsMean')
        return global_signed, global_abs, phi_mat


# Baselines
def compute_kernel_shap(model, X_train, X_test):
    try:
        import shap
        print("\nComputing Kernel SHAP (baseline) ...")
        bg  = shap.kmeans(X_train.values, min(100, len(X_train)))
        exp = shap.KernelExplainer(lambda x: model.predict_proba(x), bg)
        sv  = exp.shap_values(X_test.values, nsamples=100)
        glob = (np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
                if isinstance(sv, list) else np.abs(sv).mean(axis=0))
        return pd.Series(glob, index=X_test.columns, name='Kernel_SHAP')
    except Exception as e:
        print(f"Kernel SHAP skipped: {e}"); return None


def model_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names,
                         name='Model_Importance')
    return None


# Plots
def plot_causal_graph(dag: nx.DiGraph, gamma: Dict[str, float]):
    G_full = dag.copy()
    G_full.add_node('SPI')
    for (u, v) in CAUSAL_GRAPH:
        if v == 'SPI':
            G_full.add_edge(u, 'SPI')

    fig, ax = plt.subplots(figsize=(12, 6))
    # Hand-placed layout for clarity
    pos = {
        'Organic Carbon':           (0.05, 0.80),
        'Nitrogen':                 (0.22, 0.80),
        'Phosphorus':               (0.38, 0.80),
        'Potassium':                (0.54, 0.80),
        'pH':                       (0.14, 0.45),
        'Zinc':                     (0.30, 0.45),
        'Electrical Conductivity':  (0.78, 0.80),
        'SPI':                      (0.44, 0.08),
    }

    feat_nodes = [n for n in G_full.nodes() if n != 'SPI']
    nx.draw_networkx_nodes(G_full, pos,
                           nodelist=feat_nodes,
                           node_color='#5DCAA5', node_size=1600, alpha=0.9, ax=ax)
    nx.draw_networkx_nodes(G_full, pos,
                           nodelist=['SPI'],
                           node_color='#7F77DD', node_size=2000, alpha=0.9, ax=ax)

    short_labels = {
        'Organic Carbon': 'OC', 'Nitrogen': 'N', 'Phosphorus': 'P',
        'Potassium': 'K', 'pH': 'pH', 'Zinc': 'ZN',
        'Electrical Conductivity': 'EC', 'SPI': 'SPI',
    }
    nx.draw_networkx_labels(G_full, pos, labels=short_labels,
                            font_size=9, font_weight='bold', ax=ax)

    # Inter-feature edges (structural)
    inter_edges = [(u, v) for u, v in CAUSAL_GRAPH if v != 'SPI']
    nx.draw_networkx_edges(G_full, pos, edgelist=inter_edges,
                           edge_color='#E08020', arrows=True, arrowsize=18,
                           width=2.0, connectionstyle='arc3,rad=0.15', ax=ax)

    # Edges to SPI
    spi_edges = [(u, v) for u, v in CAUSAL_GRAPH if v == 'SPI']
    nx.draw_networkx_edges(G_full, pos, edgelist=spi_edges,
                           edge_color='#3B6EA5', arrows=True, arrowsize=18,
                           width=2.0, connectionstyle='arc3,rad=0.05', ax=ax)

    # Annotate gamma_i on SPI edges
    for (u, v) in spi_edges:
        if u in gamma:
            x0, y0 = pos[u]; x1, y1 = pos['SPI']
            ax.text((x0+x1)/2 + 0.03, (y0+y1)/2,
                    f'g={gamma[u]:.3f}',
                    fontsize=7, color='#0C447C', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white',
                              alpha=0.85, ec='#B5D4F4', lw=0.7))

    from matplotlib.lines import Line2D
    legend = [
        Line2D([0],[0], color='#E08020', lw=2, label='Structural (OC→pH→ZN)'),
        Line2D([0],[0], color='#3B6EA5', lw=2, label='Direct effect on SPI'),
    ]
    ax.legend(handles=legend, loc='lower right', fontsize=8)
    ax.set_title(
        'Observed-feature causal graph (Soil SPI DGP)\n'
        'Latent quality variable not shown  |  g = gamma_i from ground_truth.csv',
        fontsize=10, fontweight='bold'
    )
    ax.axis('off'); fig.tight_layout()
    return fig


def plot_attributions(df: pd.DataFrame):
    n_cols = df.shape[1]
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 6), sharey=False)
    if n_cols == 1: axes = [axes]
    for ax, col in zip(axes, df.columns):
        vals   = df[col].values
        colors = ['#3B8BD4' if v >= 0 else '#E24B4A' for v in vals]
        bars   = ax.barh(df.index.tolist(), vals, color=colors,
                         edgecolor='white', height=0.55)
        ax.axvline(0, color='#888', linewidth=0.8, linestyle='--')
        for bar, val in zip(bars, vals):
            ax.text(val + (0.0005 if val >= 0 else -0.0005),
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center',
                    ha='left' if val >= 0 else 'right', fontsize=8)
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel('Attribution score')
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(
        'Feature Attribution — Soil SPI (known causal graph, ground-truth gamma_i)',
        fontsize=11, fontweight='bold', y=1.01
    )
    fig.tight_layout()
    return fig


# Main
def main():
    print("\n" + "="*70)
    print("CAUSAL SHAP — Soil SPI (known causal graph, ground-truth gamma_i)")
    print("="*70 + "\n")

    # Load NPZ
    try:
        data = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X_train = pd.DataFrame(data['X_train'], columns=list(data['feature_names']))
    X_test  = pd.DataFrame(data['X_test'],  columns=list(data['feature_names']))
    y_train = pd.Series(data['y_train'].astype(int))
    y_test  = pd.Series(data['y_test'].astype(int))
    print(f"Loaded: X_train={X_train.shape}, X_test={X_test.shape}")
    print(f"Features: {list(X_train.columns)}")

    # gamma_i from exact ground truth (full dataset, unscaled)
    try:
        gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'], verbose=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['GROUND_TRUTH_PATH']}' not found. "
              "Place ground_truth.csv in the working directory."); return

    # Scale for model training and SEM fitting
    # gamma is derived from unscaled contributions - scaling only affects SEM internals
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    Xte = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    # Train model
    n_classes = int(len(np.unique(y_train)))
    print(f"Training model ({n_classes} classes: Low/Medium/High SPI) ...")
    if HAVE_XGB:
        model = XGBClassifier(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
            objective='multi:softprob', num_class=n_classes,
        )
    else:
        model = RandomForestClassifier(
            n_estimators=160, max_depth=6,
            random_state=CONFIG['RANDOM_SEED'])
    model.fit(Xtr.values, y_train.values)
    y_pred = model.predict(Xte.values)
    print(f"Accuracy={accuracy_score(y_test, y_pred):.4f}  "
          f"MacroF1={f1_score(y_test, y_pred, average='macro'):.4f}")

    # Build DAG and save graph
    dag   = build_known_dag()
    fig_g = plot_causal_graph(dag, gamma)
    fig_g.savefig('causal_graph_soil.png', dpi=150, bbox_inches='tight')
    print("Saved causal_graph_soil.png")

    # Causal SHAP
    cshap      = CausalSHAP(model=model, dag=dag, gamma=gamma,
                             X_train=Xtr, y_train=y_train)
    global_signed, global_abs, local_phi = cshap.explain_global(Xte, k=CONFIG['GLOBAL_K'])

    print("\nGlobal Causal SHAP — signed mean (direction + cancellation):")
    print(global_signed.sort_values(key=abs, ascending=False).to_string())

    print("\nGlobal Causal SHAP — mean absolute attribution (importance):")
    print(global_abs.sort_values(ascending=False).to_string())

    local_phi.to_csv('local_causal_shap_soil.csv')
    print("Saved local_causal_shap_soil.csv")

    # Baselines
    kshap = compute_kernel_shap(model, Xtr, Xte)
    mimp  = model_importance(model, list(Xtr.columns))

    # Save
    cols = {
        'Causal_SHAP_Signed':  global_signed,
        'Causal_SHAP_AbsMean': global_abs,
    }
    if kshap is not None: cols['Kernel_SHAP']     = kshap
    if mimp  is not None: cols['Model_Importance'] = mimp
    df = pd.DataFrame(cols)
    df.to_csv('attribution_scores_soil.csv')
    print("\n" + df.round(6).to_string())
    print("\nSaved attribution_scores_soil.csv")

    fig_a = plot_attributions(df)
    fig_a.savefig('attribution_comparison_soil.png', dpi=150, bbox_inches='tight')
    print("Saved attribution_comparison_soil.png")


if __name__ == '__main__':
    main()


CAUSAL SHAP — Soil SPI (known causal graph, ground-truth gamma_i)

Loaded: X_train=(4000, 7), X_test=(1000, 7)
Features: [np.str_('Organic Carbon'), np.str_('Nitrogen'), np.str_('Phosphorus'), np.str_('Potassium'), np.str_('pH'), np.str_('Zinc'), np.str_('Electrical Conductivity')]

gamma_i derivation from exact ground-truth contributions
  Feature                        E[|net|]    CV(%)    gamma_i
  ----------------------------------------------------------
  Organic Carbon                   6.5455    84.5%   0.062171
  Nitrogen                        42.2350    55.2%   0.401157
  Phosphorus                       8.0212    41.8%   0.076187
  Potassium                       27.2443    51.0%   0.258773
  pH                               9.3296   104.8%   0.088614
  Zinc                             1.5475    91.9%   0.014698
  Electrical Conductivity         10.3598    66.6%   0.098400
  ----------------------------------------------------------
  Total                          105.282

Kernel SHAP skipped: Data must be 1-dimensional, got ndarray of shape (7, 3) instead

                         Causal_SHAP_Signed  Causal_SHAP_AbsMean  Model_Importance
Organic Carbon                     0.012490             0.047299          0.114053
Nitrogen                           0.146858             0.269304          0.250705
Phosphorus                        -0.003150             0.017032          0.066715
Potassium                          0.333133             0.340118          0.251575
pH                                 0.038116             0.056000          0.186356
Zinc                              -0.000448             0.000988          0.034015
Electrical Conductivity            0.035450             0.040853          0.096580

Saved attribution_scores_soil.csv
Saved attribution_comparison_soil.png


In [ ]:
"""
Causal SHAP for Soil SPI dataset — REGRESSION version.

Identical to causal_shap_soil.py (classification) except:

  Changed
  -------
  - Model         : XGBRegressor / RandomForestRegressor
  - Target        : SPI_CONTINUOUS (float) from data_complete.csv
  - Data loading  : X and y both from data_complete.csv with 80/20 split
                    (NPZ only stores class labels, not continuous y)
  - _Ef()         : scalar E[f(X)] = mean(model.predict(X_train))
  - _vc()         : model.predict() → scalar, no class axis
  - explain_instance(): f(x) is scalar predicted SPI
  - Metrics       : RMSE, MAE, R²
  - Kernel SHAP   : uses model.predict (returns n_samples x n_features array)
  - Output cols   : GroundTruth_Gamma | Causal_SHAP_Signed | Causal_SHAP_AbsMean
  - Global metrics: Pearson r, Spearman r, MAE between
                    Causal_SHAP_AbsMean (normalised) and GroundTruth_Gamma

  Unchanged
  ---------
  - CAUSAL_GRAPH, FEATURE_NAMES, CAUSAL_GRAPH
  - compute_gamma()  — model-agnostic, derived from ground_truth.csv
  - build_known_dag()
  - LinearMechanisms (SEM)
  - Shapley MC loop, Eq. 10 weighting, Eq. 11 normalisation
  - explain_global() returning global_signed and global_abs
  - plot_causal_graph(), plot_attributions()

gamma_i derivation: soil_gamma_derivation.md (unchanged).
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import networkx as nx

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# Config
CONFIG = {
    'DATA_PATH':         '../datasets/data_complete.csv',
    'GROUND_TRUTH_PATH': '../datasets/ground_truth.csv',
    'TEST_SIZE':         0.20,      # 80/20 -> same 4000/1000 sizes as NPZ
    'MAX_PERM':    200,
    'M_RESAMPLES': 64,
    'GLOBAL_K':    30,
    'RANDOM_SEED': 42,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

FEATURE_NAMES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity',
]

COL_MAP = {   # data_complete.csv short names -> long feature names (NPZ convention)
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}

#  CAUSAL GRAPH  (unchanged from classification version)
CAUSAL_GRAPH: List[Tuple[str, str]] = [
    ('Organic Carbon', 'pH'),
    ('pH',             'Zinc'),
    ('Organic Carbon',           'SPI'),
    ('Nitrogen',                 'SPI'),
    ('Phosphorus',               'SPI'),
    ('Potassium',                'SPI'),
    ('pH',                       'SPI'),
    ('Zinc',                     'SPI'),
    ('Electrical Conductivity',  'SPI'),
]

#  GAMMA COMPUTATION  (unchanged - model-agnostic)
def compute_gamma(ground_truth_path: str, verbose: bool = True) -> Dict[str, float]:
    gt = pd.read_csv(ground_truth_path)

    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })

    mean_abs = net.abs().mean()
    cv       = net.std() / net.abs().mean()
    total    = mean_abs.sum()
    gamma    = (mean_abs / total).to_dict()

    if verbose:
        print("\n" + "=" * 68)
        print("gamma_i derivation from exact ground-truth contributions")
        print("=" * 68)
        print(f"  {'Feature':<28} {'E[|net|]':>10} {'CV(%)':>8} {'gamma_i':>10}")
        print("  " + "-" * 58)
        for f in FEATURE_NAMES:
            print(f"  {f:<28} {mean_abs[f]:>10.4f} "
                  f"{100*cv[f]:>7.1f}% {gamma[f]:>10.6f}")
        print("  " + "-" * 58)
        print(f"  {'Total':<28} {total:>10.4f} "
              f"{'':>8} {sum(gamma.values()):>10.4f}")
        print("=" * 68)
        print("  CV 40-105%: mean operating-point approximation NOT valid.\n")

    return gamma

#  BUILD FEATURE-ONLY DAG  (unchanged)
def build_known_dag() -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in CAUSAL_GRAPH:
        if v != 'SPI':
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G), "CAUSAL_GRAPH contains a cycle!"
    edges  = [(u, v) for u, v in CAUSAL_GRAPH if v != 'SPI']
    to_spi = [u for u, v in CAUSAL_GRAPH if v == 'SPI']
    print(f"Observed-feature DAG: {G.number_of_nodes()} nodes, "
          f"inter-feature edges {edges}.")
    print(f"  Direct edges to SPI: {to_spi}")
    print("  Note: latent quality variable (-> OC, N, P, K, ZN_base) "
          "is not represented in the observed DAG.")
    return G

#  LINEAR-GAUSSIAN SEM  (unchanged)
class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag           = dag
        self.feature_names = feature_names
        self.mechanisms: Dict[str, dict] = {}
        self.topo: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        self.topo = list(nx.topological_sort(self.dag))
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            mech  = self.mechanisms[var]
            sigma = mech['sigma']
            if not mech['parents']:
                mu = mech['intercept']
            else:
                pvals = np.array([x_new.get(p, float(x_obs[p]))
                                  for p in mech['parents']])
                mu = mech['intercept'] + pvals @ mech['coef']
            x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new

# Causal shap - regression
#  Three changes vs the classification CausalSHAP class:
#    1. _Ef()  -> scalar E[f(X)] = mean(model.predict(X_train))
#    2. _vc()  -> model.predict()[0], no class axis
#    3. explain_instance() -> f(x) = model.predict(x)[0], scalar
#  Everything else (Eq. 10 weighting, Eq. 11 normalisation,
#  explain_global returning global_signed and global_abs) is identical.
class CausalSHAP:
    """
    Causal SHAP — regression variant.
    vc(S) and phi_i operate on scalar predicted SPI values.
    """

    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float],
                 X_train: pd.DataFrame, y_train: pd.Series):
        self.model    = model
        self.dag      = dag
        self.gamma    = gamma
        self.X_train  = X_train
        self.features = list(X_train.columns)
        self._Ef_val  = None          # scalar cache

        print("\nFitting linear-Gaussian SEM on known DAG ...")
        self.sem = LinearMechanisms(dag, self.features).fit(X_train)
        print("SEM fitted. Ready.\n")
        print("gamma_i (causal weights):")
        for f in self.features:
            print(f"  {f:<30} gamma = {self.gamma[f]:.6f}")

    def _Ef(self) -> float:
        if self._Ef_val is None:
            self._Ef_val = float(self.model.predict(self.X_train.values).mean())
        return self._Ef_val

    def _vc(self, S: Set[str], x: pd.Series, M: int) -> float:
        total = 0.0
        for _ in range(M):
            x_s   = self.sem.sample_intervention(S, x)
            X_mat = pd.DataFrame([x_s], columns=self.features).values
            total += float(self.model.predict(X_mat)[0])
        return total / M

    def explain_instance(self, x: pd.Series,
                         n_perms: int = None, M: int = None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M       is None: M       = CONFIG['M_RESAMPLES']

        fx = float(self.model.predict(
            pd.DataFrame([x], columns=self.features).values)[0])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, M)
                phi[self.features.index(feat)] += \
                    self.gamma[feat] * (v_curr - v_prev)    # Eq. 10
                v_prev, S = v_curr, S_next
        phi /= n_perms

        sum_phi = float(np.sum(phi))
        if abs(sum_phi) > 1e-12:
            phi = phi * ((fx - self._Ef()) / sum_phi)      # Eq. 11

        return pd.Series(phi, index=self.features, name='Causal_SHAP')

    # ── unchanged ─────────────────────────────────────────────────────────
    def explain_global(self, X_test: pd.DataFrame, k: int = None):
        """
        Returns:
          global_signed  — mean(phi_i)    direction + cancellation
          global_abs     — mean(|phi_i|)  magnitude / importance
        """
        if k is None: k = CONFIG['GLOBAL_K']
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nGlobal Causal SHAP over {k} test instances ...")
        results = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            results.append(self.explain_instance(
                X_test.iloc[i],
                n_perms=max(10, CONFIG['MAX_PERM'] // 4),
            ))
        phi_mat       = pd.concat(results, axis=1)
        global_signed = phi_mat.mean(axis=1).rename('Causal_SHAP_Signed')
        global_abs    = phi_mat.abs().mean(axis=1).rename('Causal_SHAP_AbsMean')
        return global_signed, global_abs

# Global ranking metrics
#  Compare Causal_SHAP_AbsMean (normalised to sum 1) vs GroundTruth_Gamma
def compute_global_metrics(gamma: Dict[str, float],
                           global_abs: pd.Series) -> Dict[str, float]:
    gt_vec   = np.array([gamma[f]             for f in FEATURE_NAMES])
    shap_raw = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
    shap_norm = shap_raw / shap_raw.sum() if shap_raw.sum() > 1e-12 else shap_raw

    mae            = float(np.mean(np.abs(shap_norm - gt_vec)))
    pearson_r, pp  = pearsonr(shap_norm, gt_vec)
    spearman_r, sp = spearmanr(shap_norm, gt_vec)

    metrics = {
        'MAE':        mae,
        'Pearson_r':  float(pearson_r),
        'Pearson_p':  float(pp),
        'Spearman_r': float(spearman_r),
        'Spearman_p': float(sp),
    }

    print("\n--- Global ranking metrics "
          "(Causal_SHAP_AbsMean normalised vs GroundTruth_Gamma) ---")
    print(f"  MAE        = {mae:.6f}")
    print(f"  Pearson r  = {pearson_r:.4f}  (p={pp:.4f})")
    print(f"  Spearman r = {spearman_r:.4f}  (p={sp:.4f})")
    return metrics

#  BASELINES  (kernel SHAP uses model.predict - regression)
def compute_kernel_shap(model, X_train, X_test):
    try:
        import shap
        print("\nComputing Kernel SHAP (baseline) ...")
        bg  = shap.kmeans(X_train.values, min(100, len(X_train)))
        exp = shap.KernelExplainer(model.predict, bg)   # predict, not predict_proba
        sv  = exp.shap_values(X_test.values, nsamples=100)
        # Regression: sv is (n_samples x n_features) - no class axis
        glob = np.abs(sv).mean(axis=0)
        return pd.Series(glob, index=X_test.columns, name='Kernel_SHAP')
    except Exception as e:
        print(f"Kernel SHAP skipped: {e}"); return None


def model_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names,
                         name='Model_Importance')
    return None

#  PLOTS  (unchanged from classification version)
def plot_causal_graph(dag: nx.DiGraph, gamma: Dict[str, float]):
    G_full = dag.copy()
    G_full.add_node('SPI')
    for (u, v) in CAUSAL_GRAPH:
        if v == 'SPI':
            G_full.add_edge(u, 'SPI')

    fig, ax = plt.subplots(figsize=(12, 6))
    pos = {
        'Organic Carbon':           (0.05, 0.80),
        'Nitrogen':                 (0.22, 0.80),
        'Phosphorus':               (0.38, 0.80),
        'Potassium':                (0.54, 0.80),
        'pH':                       (0.14, 0.45),
        'Zinc':                     (0.30, 0.45),
        'Electrical Conductivity':  (0.78, 0.80),
        'SPI':                      (0.44, 0.08),
    }
    feat_nodes = [n for n in G_full.nodes() if n != 'SPI']
    nx.draw_networkx_nodes(G_full, pos, nodelist=feat_nodes,
                           node_color='#5DCAA5', node_size=1600, alpha=0.9, ax=ax)
    nx.draw_networkx_nodes(G_full, pos, nodelist=['SPI'],
                           node_color='#7F77DD', node_size=2000, alpha=0.9, ax=ax)
    short_labels = {
        'Organic Carbon':'OC', 'Nitrogen':'N', 'Phosphorus':'P',
        'Potassium':'K', 'pH':'pH', 'Zinc':'ZN',
        'Electrical Conductivity':'EC', 'SPI':'SPI',
    }
    nx.draw_networkx_labels(G_full, pos, labels=short_labels,
                            font_size=9, font_weight='bold', ax=ax)
    inter_edges = [(u, v) for u, v in CAUSAL_GRAPH if v != 'SPI']
    nx.draw_networkx_edges(G_full, pos, edgelist=inter_edges,
                           edge_color='#E08020', arrows=True, arrowsize=18,
                           width=2.0, connectionstyle='arc3,rad=0.15', ax=ax)
    spi_edges = [(u, v) for u, v in CAUSAL_GRAPH if v == 'SPI']
    nx.draw_networkx_edges(G_full, pos, edgelist=spi_edges,
                           edge_color='#3B6EA5', arrows=True, arrowsize=18,
                           width=2.0, connectionstyle='arc3,rad=0.05', ax=ax)
    for (u, v) in spi_edges:
        if u in gamma:
            x0, y0 = pos[u]; x1, y1 = pos['SPI']
            ax.text((x0+x1)/2 + 0.03, (y0+y1)/2,
                    f'g={gamma[u]:.3f}',
                    fontsize=7, color='#0C447C', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white',
                              alpha=0.85, ec='#B5D4F4', lw=0.7))
    from matplotlib.lines import Line2D
    ax.legend(handles=[
        Line2D([0],[0], color='#E08020', lw=2, label='Structural (OC→pH→ZN)'),
        Line2D([0],[0], color='#3B6EA5', lw=2, label='Direct effect on SPI'),
    ], loc='lower right', fontsize=8)
    ax.set_title(
        'Observed-feature causal graph (Soil SPI DGP) — regression\n'
        'Latent quality variable not shown  |  g = gamma_i from ground_truth.csv',
        fontsize=10, fontweight='bold'
    )
    ax.axis('off'); fig.tight_layout()
    return fig


def plot_attributions(df: pd.DataFrame):
    n_cols = df.shape[1]
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 6), sharey=False)
    if n_cols == 1: axes = [axes]
    for ax, col in zip(axes, df.columns):
        vals   = df[col].values
        colors = ['#3B8BD4' if v >= 0 else '#E24B4A' for v in vals]
        bars   = ax.barh(df.index.tolist(), vals, color=colors,
                         edgecolor='white', height=0.55)
        ax.axvline(0, color='#888', linewidth=0.8, linestyle='--')
        for bar, val in zip(bars, vals):
            ax.text(val + (0.0005 if val >= 0 else -0.0005),
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center',
                    ha='left' if val >= 0 else 'right', fontsize=8)
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel('Attribution score')
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(
        'Feature Attribution — Soil SPI Regression\n'
        '(known causal graph, ground-truth gamma_i)',
        fontsize=11, fontweight='bold', y=1.01
    )
    fig.tight_layout()
    return fig

# Main
def main():
    print("\n" + "="*70)
    print("CAUSAL SHAP — Soil SPI REGRESSION (SPI_CONTINUOUS target)")
    print("="*70 + "\n")

    # ── Load X and y from data_complete.csv ──────────────────────────────
    # The NPZ stores only class labels; continuous y must come from the CSV.
    # Both use the same 5000-row dataset; 80/20 split matches NPZ sizes.
    try:
        dc = pd.read_csv(CONFIG['DATA_PATH']).rename(columns=COL_MAP)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X = dc[FEATURE_NAMES]
    y = dc['SPI_CONTINUOUS']
    print(f"Loaded: {X.shape[0]} rows, "
          f"SPI_CONTINUOUS [{y.min():.1f}, {y.max():.1f}] "
          f"mean={y.mean():.2f} std={y.std():.2f}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=CONFIG['TEST_SIZE'],
        random_state=CONFIG['RANDOM_SEED'],
        shuffle=True,
    )
    print(f"Split: train={len(X_train)}, test={len(X_test)}\n")

    # ── gamma_i (unchanged - derived from ground_truth.csv) ──────────────
    try:
        gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'], verbose=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['GROUND_TRUTH_PATH']}' not found."); return

    # ── Scale ─────────────────────────────────────────────────────────────
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train),
                       columns=FEATURE_NAMES, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test),
                       columns=FEATURE_NAMES, index=X_test.index)

    # ── Train regression model ────────────────────────────────────────────
    print("Training regression model ...")
    if HAVE_XGB:
        model = XGBRegressor(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
        )
        print("Using XGBRegressor")
    else:
        model = RandomForestRegressor(
            n_estimators=160, max_depth=6,
            random_state=CONFIG['RANDOM_SEED'])
        print("Using RandomForestRegressor")

    model.fit(Xtr.values, y_train.values)
    y_pred = model.predict(Xte.values)
    rmse = float(np.sqrt(mean_squared_error(y_test.values, y_pred)))
    mae  = float(mean_absolute_error(y_test.values, y_pred))
    r2   = float(r2_score(y_test.values, y_pred))
    print(f"RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}\n")

    # ── Build DAG and save graph ──────────────────────────────────────────
    dag   = build_known_dag()
    fig_g = plot_causal_graph(dag, gamma)
    fig_g.savefig('causal_graph_soil_regression.png', dpi=150, bbox_inches='tight')
    print("Saved causal_graph_soil_regression.png")

    # ── Causal SHAP ───────────────────────────────────────────────────────
    cshap = CausalSHAP(model=model, dag=dag, gamma=gamma,
                       X_train=Xtr, y_train=y_train)
    global_signed, global_abs = cshap.explain_global(Xte, k=CONFIG['GLOBAL_K'])

    print("\nGlobal Causal SHAP — signed mean (direction + cancellation):")
    print(global_signed.sort_values(key=abs, ascending=False).to_string())
    print("\nGlobal Causal SHAP — mean absolute attribution (importance):")
    print(global_abs.sort_values(ascending=False).to_string())

    # ── Global ranking metrics ────────────────────────────────────────────
    metrics = compute_global_metrics(gamma, global_abs)

    # ── Baselines ─────────────────────────────────────────────────────────
    kshap = compute_kernel_shap(model, Xtr, Xte)
    mimp  = model_importance(model, FEATURE_NAMES)

    # ── Output table - GT gamma first column ─────────────────────────────
    gt_gamma = pd.Series(gamma, name='GroundTruth_Gamma')
    cols = {
        'GroundTruth_Gamma':   gt_gamma,
        'Causal_SHAP_Signed':  global_signed,
        'Causal_SHAP_AbsMean': global_abs,
    }
    if kshap is not None: cols['Kernel_SHAP']     = kshap
    if mimp  is not None: cols['Model_Importance'] = mimp
    df = pd.DataFrame(cols)
    df.to_csv('attribution_scores_soil_regression.csv')
    print("\n" + df.round(6).to_string())
    print("\nSaved attribution_scores_soil_regression.csv")

    # ── Plots ─────────────────────────────────────────────────────────────
    fig_a = plot_attributions(df)
    fig_a.savefig('attribution_comparison_soil_regression.png',
                  dpi=150, bbox_inches='tight')
    print("Saved attribution_comparison_soil_regression.png")


if __name__ == '__main__':
    pass
main()  


CAUSAL SHAP — Soil SPI REGRESSION (SPI_CONTINUOUS target)

Loaded: 5000 rows, SPI_CONTINUOUS [71.9, 302.2] mean=163.24 std=32.66
Split: train=4000, test=1000


gamma_i derivation from exact ground-truth contributions
  Feature                        E[|net|]    CV(%)    gamma_i
  ----------------------------------------------------------
  Organic Carbon                   6.5455    84.5%   0.062171
  Nitrogen                        42.2350    55.2%   0.401157
  Phosphorus                       8.0212    41.8%   0.076187
  Potassium                       27.2443    51.0%   0.258773
  pH                               9.3296   104.8%   0.088614
  Zinc                             1.5475    91.9%   0.014698
  Electrical Conductivity         10.3598    66.6%   0.098400
  ----------------------------------------------------------
  Total                          105.2828              1.0000
  CV 40-105%: mean operating-point approximation NOT valid.

Training regression model ...
Using XGBRe


                         GroundTruth_Gamma  Causal_SHAP_Signed  Causal_SHAP_AbsMean  Kernel_SHAP  Model_Importance
Organic Carbon                    0.062171           -0.547834             1.094862     3.730218          0.042207
Nitrogen                          0.401157           -7.897459            13.082366    15.440427          0.512452
Phosphorus                        0.076187           -0.455914             0.553324     2.509820          0.018255
Potassium                         0.258773           -1.863127             9.563682    11.453389          0.218761
pH                                0.088614           -0.103258             1.617331     7.769696          0.162176
Zinc                              0.014698            0.003300             0.033589     0.785148          0.005072
Electrical Conductivity           0.098400            0.258078             0.997925     4.485413          0.041078

Saved attribution_scores_soil_regression.csv
Saved attribution_comparison_soil

In [ ]:
"""
causal_shap_soil_robustness.py
==============================
Robustness analysis of Causal SHAP attributions under SEM edge-weight
perturbation for the Soil SPI REGRESSION dataset.

PRECISE CLAIM TESTED
---------------------
  "Causal SHAP attributions for soil SPI prediction are sensitive to errors
   in SEM edge-weight estimation, even when the KNOWN causal graph structure
   (OC->pH->ZN, all features->SPI) is held fixed and correct.
   This isolates SEM-parameter sensitivity from graph-structure uncertainty."

WHAT THIS DOES NOT TEST
------------------------
  Missing edges, wrong edge directions, or extra edges.

KEY FIXES vs INITIAL VERSION (rescale scalar = 101 problem)
------------------------------------------------------------
  FIX 1 — ANTITHETIC PERMUTATIONS
    For every permutation pi, also evaluate reverse(pi).
    This costs nothing extra (same n_perms budget, 2x effective samples)
    and is the primary reason the scalar was inflated: with n_perms=30 and
    7 features, marginal contributions partially cancel in expectation.
    Antithetic pairs are negatively correlated => variance halved for free.

  FIX 2 — INCREASED MC BUDGET
    n_perms   : 30  -> 200   (need >= 4*n_feat*(n_feat-1) = 168 for 7 features)
    M_per_sem : 16  -> 32    (more stable v(S) estimates per coalition)
    k_instances: 5  -> 10    (more robust global mean)

  FIX 3 — SCALAR GUARD WITH DIAGNOSTICS
    If |scalar| > 20 after antithetic correction, the run is flagged invalid
    and the trial is skipped in the MAD average rather than inflating it.
    This prevents a noisy outlier from corrupting the ARM 1 results.

  FIX 4 — BASELINE CONSISTENCY CHECK
    Before running ARM 1/2, we verify that phi0_mean sums to
    (f(x) - E[f(X)]) within 5% for each instance. Failure => abort with
    a clear message to increase n_perms further.

TWO EXPERIMENTAL ARMS
---------------------
  ARM 1 — Adversarial / worst-case perturbation
    Baseline SEM weights fit on full X_train.
    For each epsilon in EPSILONS:
      wij' = wij * (1 + epsilon * delta),   delta ~ Uniform(-1, 1)
      Repeat N_TRIALS times (different delta draws, same instances).
      Measure: MAD(phi_eps, phi_0) and Spearman rho of feature rankings.
    Find eps* = first epsilon where noise-corrected MAD > 5% of |phi_0|.

  ARM 2 — Estimation noise (realistic / constructive)
    SEM weights refit on subsets of X_train of size n.
    Measure: MAD(phi_n, phi_full) as a function of n.
    Model, graph, and test instances are FROZEN.

KEY VALIDITY CONTROLS
---------------------
  1. SHARED RANDOM SEEDS  — instance_indices, perm_seeds, noise_seeds
     generated ONCE, reused for every run. Only SEM coefficients change.
  2. epsilon=0 NOISE FLOOR  — pure MC estimator noise; all reported MAD
     is noise-corrected: MAD_corr = max(0, MAD(eps) - MAD(0)).
  3. MONOTONICITY CHECK  — corrected MAD must increase with epsilon.
  4. WEAK-EDGE PROTECTION  — edges |wij| < 1e-3 are not perturbed.
  5. GAMMA FROM GROUND TRUTH  — gamma_i fixed from ground_truth.csv,
     independent of the SEM.
  6. ANTITHETIC PERMUTATIONS  — each perm seed generates both pi and
     reverse(pi), halving estimator variance at zero extra cost.

OUTPUTS
-------
  arm1_adversarial_soil.png
  arm2_estimation_soil.png
  gamma_vs_sensitivity_soil.png
  arm1_scores_soil.csv
  arm2_scores_soil.csv
"""

import copy
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx

from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from typing import Dict, List, Optional, Set, Tuple

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"figure.dpi": 120})

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# Configuration

CFG = {
    # Paths
    "data_path":         "../datasets/data_complete.csv",
    "ground_truth_path": "../datasets/ground_truth.csv",

    # Data
    "test_size":         0.20,
    "random_seed":       42,

    "n_perms":           200,   # was 30;  need ≥ 4*7*6=168 for 7 features
    "M_per_sem":         32,    # was 16;  more stable coalition estimates
    "k_instances":       10,    # was 5;   more robust global mean
    "use_antithetic":    True,
    "scalar_guard":      20.0,

    # ARM 1
    "epsilons":          [0.0, 0.05, 0.10, 0.30, 0.50, 1.0],
    "n_trials":          3,
    "breakdown_pct":     0.05,

    # ARM 2
    "subsample_sizes":   [50, 100, 250, 500, 750],
}


FEATURE_NAMES = [
    "Organic Carbon", "Nitrogen", "Phosphorus", "Potassium",
    "pH", "Zinc", "Electrical Conductivity",
]

COL_MAP = {
    "OC": "Organic Carbon", "N": "Nitrogen",  "P": "Phosphorus",
    "K":  "Potassium",      "PH": "pH",        "ZN": "Zinc",
    "EC": "Electrical Conductivity",
}

CAUSAL_GRAPH: List[Tuple[str, str]] = [
    ("Organic Carbon", "pH"),
    ("pH",             "Zinc"),
    ("Organic Carbon",          "SPI"),
    ("Nitrogen",                "SPI"),
    ("Phosphorus",              "SPI"),
    ("Potassium",               "SPI"),
    ("pH",                      "SPI"),
    ("Zinc",                    "SPI"),
    ("Electrical Conductivity", "SPI"),
]

WEAK_EDGE_THRESHOLD = 1e-3


# 1. data loading

def load_soil_data(cfg: dict):
    dc = pd.read_csv(cfg["data_path"]).rename(columns=COL_MAP)
    X  = dc[FEATURE_NAMES]
    y  = dc["SPI_CONTINUOUS"]
    print(f"Loaded {len(X)} rows. SPI range [{y.min():.2f}, {y.max():.2f}]  "
          f"mean={y.mean():.2f}  std={y.std():.2f}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg["test_size"],
        random_state=cfg["random_seed"], shuffle=True,
    )
    scaler = StandardScaler()
    Xtr = pd.DataFrame(
        scaler.fit_transform(X_train), columns=FEATURE_NAMES, index=X_train.index
    )
    Xte = pd.DataFrame(
        scaler.transform(X_test), columns=FEATURE_NAMES, index=X_test.index
    )
    print(f"Train: {len(Xtr)}   Test: {len(Xte)}")
    return Xtr, Xte, y_train, y_test


# 2. gamma - from ground truth csv

def load_gamma(ground_truth_path: str, verbose: bool = True) -> Dict[str, float]:
    gt  = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        "Organic Carbon":          gt["OC_base"]  - gt["OC_penalty"],
        "Nitrogen":                gt["N_base"]   - gt["N_penalty"],
        "Phosphorus":              gt["P_base"]   - gt["P_penalty"],
        "Potassium":               gt["K_base"]   - gt["K_penalty"],
        "pH":                      -gt["PH_penalty"],
        "Zinc":                    gt["ZN_base"]  - gt["ZN_penalty"],
        "Electrical Conductivity": -gt["EC_penalty"],
    })
    mean_abs = net.abs().mean()
    total    = mean_abs.sum()
    gamma    = (mean_abs / total).to_dict()

    if verbose:
        print("\ngamma_i from ground_truth.csv:")
        for f in FEATURE_NAMES:
            print(f"  {f:<30}  gamma = {gamma[f]:.6f}")
        print(f"  {'sum':<30}  {sum(gamma.values()):.6f}\n")
    return gamma


# 3. build known dag

def build_known_dag() -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for u, v in CAUSAL_GRAPH:
        if v != "SPI":
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G)
    return G


# 4. sem - fit and perturb

class SEM:
    """
    Linear-Gaussian SEM for the soil feature DAG.

    ARM 1 PERTURBATION:
        wij' = wij * (1 + epsilon * delta),   delta ~ U(-1, 1)
    Edges with |beta_ij| < WEAK_EDGE_THRESHOLD are not perturbed.
    """

    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = dag
        self.features = list(features)
        self.mechs:   Dict[str, dict] = {}
        self.topo:    List[str]       = []

    def fit(self, X: pd.DataFrame,
            subsample_idx: Optional[np.ndarray] = None) -> "SEM":
        Xf = X.iloc[subsample_idx] if subsample_idx is not None else X
        self.topo = list(nx.topological_sort(self.dag))
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechs[var] = {
                    "parents":   [],
                    "intercept": float(Xf[var].mean()),
                    "coef":      np.array([]),
                    "sigma":     float(max(Xf[var].std(ddof=0), 1e-9)),
                }
            else:
                lr    = LinearRegression().fit(Xf[parents].values, Xf[var].values)
                resid = Xf[var].values - lr.predict(Xf[parents].values)
                self.mechs[var] = {
                    "parents":   parents,
                    "intercept": float(lr.intercept_),
                    "coef":      lr.coef_.astype(float).copy(),
                    "sigma":     float(max(np.std(resid, ddof=0), 1e-9)),
                }
        return self

    def perturbed_mechs(self, epsilon: float,
                        pert_rng: np.random.Generator) -> Dict[str, dict]:
        mechs = {}
        for var, m in self.mechs.items():
            mc = copy.deepcopy(m)
            if mc["coef"].size > 0 and epsilon > 0.0:
                new_coef = mc["coef"].copy()
                for k, c in enumerate(mc["coef"]):
                    if abs(c) >= WEAK_EDGE_THRESHOLD:
                        delta       = pert_rng.uniform(-1.0, 1.0)
                        new_coef[k] = c * (1.0 + epsilon * delta)
                mc["coef"] = new_coef
            mechs[var] = mc
        return mechs

    def sample_intervention(self, S: Set[str], x_obs: pd.Series,
                            mechs: Dict[str, dict],
                            noise_rng: np.random.Generator) -> Dict[str, float]:
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            m   = mechs[var]
            mu  = m["intercept"]
            sig = m["sigma"]
            if m["parents"]:
                pv  = np.array([x_new.get(p, float(x_obs[p])) for p in m["parents"]])
                mu += float(pv @ m["coef"])
            x_new[var] = mu if sig < 1e-9 else float(noise_rng.normal(mu, sig))
        return x_new


# 5. value function - regression

def _predict_scalar(model, X_mat: np.ndarray) -> float:
    return float(model.predict(X_mat)[0])


def _E_f(model, X_train: pd.DataFrame) -> float:
    return float(model.predict(X_train.values).mean())


def _v_coalition(S: Set[str], x_obs: pd.Series,
                 features: List[str],
                 sems: List[SEM], mechs_list: List[Dict],
                 model, M_per_sem: int,
                 noise_seeds: List[int]) -> float:
    """
    v(S) = E[f(X) | do(X_S = x_S)] estimated by MC.
    noise_seeds are SHARED across all runs so only SEM coefficients vary.
    """
    total, count, seed_idx = 0.0, 0, 0
    for sem, mechs in zip(sems, mechs_list):
        for _ in range(M_per_sem):
            noise_rng = np.random.default_rng(
                noise_seeds[seed_idx % len(noise_seeds)]
            )
            seed_idx += 1
            xs    = sem.sample_intervention(S, x_obs, mechs, noise_rng)
            X_mat = np.array([[xs[f] for f in features]], dtype=float)
            total += _predict_scalar(model, X_mat)
            count += 1
    return total / count if count > 0 else 0.0


# 6. causal shap - antithetic permutation estimator

def _shap_instance(x: pd.Series, features: List[str],
                   sems: List[SEM], mechs_list: List[Dict],
                   gamma: Dict[str, float], model,
                   Ef: float, n_perms: int, M_per_sem: int,
                   perm_seeds: List[int], noise_seeds: List[int],
                   use_antithetic: bool = True,
                   scalar_guard: float = 20.0) -> Optional[np.ndarray]:
    """
    Causal SHAP for one instance using the antithetic permutation estimator.

    ANTITHETIC PERMUTATIONS (FIX 1)
    --------------------------------
    For each permutation seed, we evaluate BOTH the forward permutation pi
    and its reverse reverse(pi).  The two estimates are negatively correlated
    because a feature that appears early in pi appears late in reverse(pi),
    so its marginal contribution is evaluated against complementary coalitions.
    Averaging the pair halves the estimator variance at zero extra MC cost
    (we reuse the same noise_seeds for both orderings).

    SCALAR GUARD (FIX 3)
    ---------------------
    After rescaling, if |scalar| > scalar_guard the instance is flagged and
    None is returned.  The caller skips it in the global mean.  This prevents
    a single poorly-converged instance from inflating MAD estimates.
    """
    X_mat_x = np.array([[x[f] for f in features]], dtype=float)
    fx       = _predict_scalar(model, X_mat_x)
    phi      = np.zeros(len(features))
    n_evals  = 0

    for perm_idx in range(n_perms):
        perm_rng    = np.random.default_rng(perm_seeds[perm_idx])
        fwd_perm    = list(perm_rng.permutation(features))
        perms_to_run = [fwd_perm, list(reversed(fwd_perm))] \
                       if use_antithetic else [fwd_perm]

        for perm in perms_to_run:
            S: Set[str] = set()
            v_prev = _v_coalition(S, x, features, sems, mechs_list,
                                  model, M_per_sem, noise_seeds)
            for feat in perm:
                S_next = S | {feat}
                v_curr = _v_coalition(S_next, x, features, sems, mechs_list,
                                      model, M_per_sem, noise_seeds)
                phi[features.index(feat)] += gamma[feat] * (v_curr - v_prev)
                v_prev, S = v_curr, S_next
            n_evals += 1

    phi /= n_evals

    s = float(np.sum(phi))
    if abs(s) < 1e-12:
        return None

    scalar = (fx - Ef) / s
    if abs(scalar) > scalar_guard:
        return None   # caller will skip this instance

    return phi * scalar


def _shap_global(X_test: pd.DataFrame, features: List[str],
                 sems: List[SEM], mechs_list: List[Dict],
                 gamma: Dict[str, float], model, Ef: float,
                 instance_indices: np.ndarray,
                 n_perms: int, M_per_sem: int,
                 perm_seeds: List[int], noise_seeds: List[int],
                 use_antithetic: bool = True,
                 scalar_guard: float = 20.0,
                 verbose: bool = False) -> np.ndarray:
    """
    Global Causal SHAP — returns (k × n_features) matrix.
    Instances where |scalar| > scalar_guard are excluded from the matrix.
    """
    phis       = []
    n_skipped  = 0
    for t, i in enumerate(instance_indices):
        if verbose:
            print(f"    instance {t+1}/{len(instance_indices)}", flush=True)
        result = _shap_instance(
            X_test.iloc[i], features, sems, mechs_list,
            gamma, model, Ef,
            n_perms, M_per_sem, perm_seeds, noise_seeds,
            use_antithetic=use_antithetic,
            scalar_guard=scalar_guard,
        )
        if result is None:
            n_skipped += 1
            if verbose:
                print(f"      [SKIP] instance {t+1} — scalar guard triggered")
        else:
            phis.append(result)

    if n_skipped > 0:
        print(f"    [INFO] {n_skipped}/{len(instance_indices)} instances skipped "
              f"(scalar > {scalar_guard}). "
              f"If > 50%, increase n_perms further.")

    if not phis:
        raise RuntimeError(
            "ALL instances failed the scalar guard. "
            "Increase n_perms or M_per_sem substantially."
        )

    return np.array(phis)


def _check_baseline_consistency(baseline_phi: np.ndarray,
                                X_test: pd.DataFrame,
                                instance_indices: np.ndarray,
                                model, Ef: float,
                                features: List[str],
                                tol: float = 0.05) -> bool:
    """
    FIX 4: Verify sum(phi_i) ≈ f(x) - E[f(X)] for each instance.
    Tolerance is tol * |f(x) - Ef|.
    Returns True if all instances pass, False otherwise.
    """
    all_pass = True
    valid_idx = range(len(baseline_phi))   # after scalar guard, all rows are valid
    for row_idx, inst_idx in zip(range(len(baseline_phi)), instance_indices[:len(baseline_phi)]):
        x     = X_test.iloc[inst_idx]
        fx    = float(model.predict(
            np.array([[x[f] for f in features]], dtype=float))[0])
        diff  = fx - Ef
        s     = float(np.sum(baseline_phi[row_idx]))
        err   = abs(s - diff)
        thresh = tol * abs(diff) if abs(diff) > 1e-8 else 1e-4
        ok    = err <= thresh
        if not ok:
            print(f"    [WARN] Instance {inst_idx}: sum(phi)={s:.4f}  "
                  f"f(x)-Ef={diff:.4f}  err={err:.4f} > {tol*100:.0f}% tol")
            all_pass = False
    return all_pass


# 7. arm 1 - adversarial epsilon sweep

def _arm1_sweep(baseline_phi: np.ndarray,
                X_test: pd.DataFrame, features: List[str],
                sems: List[SEM], gamma: Dict[str, float],
                model, Ef: float,
                instance_indices: np.ndarray,
                perm_seeds: List[int], noise_seeds: List[int],
                master_rng: np.random.Generator,
                cfg: dict) -> List[dict]:
    phi0_mean = baseline_phi.mean(axis=0)
    rows      = []

    for eps in cfg["epsilons"]:
        print(f"  eps={eps:.3f} ...", end="", flush=True)
        trial_mads, trial_rhos, trial_feat_diffs = [], [], []

        for _ in range(cfg["n_trials"]):
            pert_rng        = np.random.default_rng(int(master_rng.integers(0, 2**32)))
            perturbed_mechs = [sem.perturbed_mechs(eps, pert_rng) for sem in sems]

            phi_eps = _shap_global(
                X_test, features, sems, perturbed_mechs, gamma,
                model, Ef,
                instance_indices=instance_indices,
                n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
                perm_seeds=perm_seeds, noise_seeds=noise_seeds,
                use_antithetic=cfg["use_antithetic"],
                scalar_guard=cfg["scalar_guard"],
            )
            phi_eps_mean = phi_eps.mean(axis=0)
            diff         = np.abs(phi_eps_mean - phi0_mean)
            trial_mads.append(float(diff.mean()))
            trial_feat_diffs.append(diff)
            rho, _ = spearmanr(np.abs(phi0_mean), np.abs(phi_eps_mean))
            trial_rhos.append(float(rho) if not np.isnan(rho) else 1.0)

        rows.append({
            "epsilon":         eps,
            "mad_mean":        float(np.mean(trial_mads)),
            "mad_std":         float(np.std(trial_mads)),
            "per_feature_mad": np.stack(trial_feat_diffs).mean(axis=0),
            "spearman_mean":   float(np.mean(trial_rhos)),
            "spearman_std":    float(np.std(trial_rhos)),
        })
        r = rows[-1]
        print(f"  MAD={r['mad_mean']:.5f} ± {r['mad_std']:.5f}"
              f"   rho={r['spearman_mean']:.4f} ± {r['spearman_std']:.4f}")

    return rows


def _check_monotonicity(rows: List[dict]) -> None:
    noise_floor = next((r["mad_mean"] for r in rows if r["epsilon"] == 0.0), 0.0)
    nonzero     = [r for r in rows if r["epsilon"] > 0.0]
    corrected   = [max(0.0, r["mad_mean"] - noise_floor) for r in nonzero]
    eps_vals    = [r["epsilon"] for r in nonzero]
    violations  = [
        (eps_vals[i-1], eps_vals[i], corrected[i-1], corrected[i])
        for i in range(1, len(corrected))
        if corrected[i] < corrected[i-1] * 0.90
    ]
    if violations:
        print("\n  WARNING: MONOTONICITY VIOLATIONS detected:")
        for v in violations:
            print(f"    eps={v[0]:.2f}->{v[1]:.2f}: MAD {v[2]:.5f}->{v[3]:.5f}")
        print("  -> Increase n_perms / M_per_sem before trusting these results.\n")
    else:
        print("\n  OK: Monotonicity holds — results are reliable.\n")


# 8. arm 2 - estimation-noise sweep

def _arm2_sweep(baseline_phi: np.ndarray,
                X_train: pd.DataFrame, X_test: pd.DataFrame,
                features: List[str], dag: nx.DiGraph,
                gamma: Dict[str, float], model, Ef: float,
                instance_indices: np.ndarray,
                perm_seeds: List[int], noise_seeds: List[int],
                master_rng: np.random.Generator,
                cfg: dict) -> List[dict]:
    phi0_mean = baseline_phi.mean(axis=0)
    n_full    = len(X_train)
    rows      = []

    for n in cfg["subsample_sizes"]:
        if n >= n_full:
            print(f"  n={n}: skipping (>= full training size {n_full})")
            continue
        print(f"  n={n} ...", end="", flush=True)

        sub_idx = master_rng.choice(n_full, size=n, replace=False)
        sems_n  = [SEM(dag, features).fit(X_train, subsample_idx=sub_idx)]

        phi_n = _shap_global(
            X_test, features, sems_n, [s.mechs for s in sems_n], gamma,
            model, Ef,
            instance_indices=instance_indices,
            n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
            perm_seeds=perm_seeds, noise_seeds=noise_seeds,
            use_antithetic=cfg["use_antithetic"],
            scalar_guard=cfg["scalar_guard"],
        )
        phi_n_mean = phi_n.mean(axis=0)
        diff       = np.abs(phi_n_mean - phi0_mean)
        rho, _     = spearmanr(np.abs(phi0_mean), np.abs(phi_n_mean))

        rows.append({
            "n":               n,
            "mad_mean":        float(diff.mean()),
            "per_feature_mad": diff,
            "spearman":        float(rho) if not np.isnan(rho) else 1.0,
        })
        print(f"  MAD={diff.mean():.5f}   rho={rows[-1]['spearman']:.4f}")

    return rows


# 9. plots

def _plot_arm1(rows: List[dict], features: List[str],
               phi0_mean: np.ndarray, cfg: dict) -> Optional[float]:
    noise_floor = next((r["mad_mean"] for r in rows if r["epsilon"] == 0.0), 0.0)
    eps_vals    = [r["epsilon"]       for r in rows]
    mad_means   = [r["mad_mean"]      for r in rows]
    mad_stds    = [r["mad_std"]       for r in rows]
    corrected   = [max(0.0, m - noise_floor) for m in mad_means]
    rho_means   = [r["spearman_mean"] for r in rows]
    rho_stds    = [r["spearman_std"]  for r in rows]

    ref       = np.abs(phi0_mean).mean()
    threshold = cfg["breakdown_pct"] * ref if ref > 1e-10 else None
    eps_star  = None
    if threshold:
        for r, c in zip(rows, corrected):
            if r["epsilon"] == 0.0: continue
            if c > threshold:
                eps_star = r["epsilon"]; break

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(
        "ARM 1: SEM Weight Robustness — Soil SPI Regression\n"
        "(known causal graph fixed; only SEM edge weights perturbed)\n"
        f"Antithetic permutations: {cfg['use_antithetic']}  |  "
        f"n_perms={cfg['n_perms']}  M_per_sem={cfg['M_per_sem']}",
        fontsize=11, fontweight="bold"
    )

    # Raw MAD
    ax = axes[0, 0]
    lo = np.array(mad_means) - np.array(mad_stds)
    hi = np.array(mad_means) + np.array(mad_stds)
    ax.plot(eps_vals, mad_means, "o-", color="#185FA5", lw=2, ms=6,
            label=f"MAD mean ({cfg['n_trials']} trials)")
    ax.fill_between(eps_vals, lo, hi, alpha=0.18, color="#185FA5", label="±1 std")
    ax.axhline(noise_floor, color="#888", ls=":", lw=1.4,
               label=f"noise floor (ε=0): {noise_floor:.5f}")
    if threshold:
        ax.axhline(threshold, color="#A32D2D", ls="--", lw=1.3,
                   label=f"5% of |φ₀|: {threshold:.5f}")
    ax.set_xlabel("ε  (relative weight perturbation)")
    ax.set_ylabel("MAD(φ_ε − φ₀)")
    ax.set_title("Raw MAD vs ε")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    # Noise-corrected MAD
    ax = axes[0, 1]
    ax.plot(eps_vals, corrected, "s-", color="#0F6E56", lw=2, ms=6,
            label="MAD − noise floor")
    if threshold:
        ax.axhline(threshold, color="#A32D2D", ls="--", lw=1.3,
                   label=f"5% of |φ₀|: {threshold:.5f}")
    if eps_star:
        ax.axvline(eps_star, color="#A32D2D", ls=":", lw=1.5,
                   label=f"ε* = {eps_star:.2f}")
    ax.set_xlabel("ε  (relative weight perturbation)")
    ax.set_ylabel("Noise-corrected MAD")
    ax.set_title(f"Corrected MAD vs ε   (ε* = {eps_star})")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    # Per-feature heatmap
    ax = axes[1, 0]
    mat   = np.stack([r["per_feature_mad"] for r in rows], axis=1)
    order = np.argsort(-mat.sum(axis=1))
    mat_s = mat[order]
    fs    = [features[i] for i in order]
    im    = ax.imshow(mat_s, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(np.arange(len(eps_vals)))
    ax.set_xticklabels([f"{e:.2f}" for e in eps_vals], rotation=30,
                       ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(fs))); ax.set_yticklabels(fs, fontsize=9)
    for i in range(mat_s.shape[0]):
        for j in range(mat_s.shape[1]):
            v = mat_s[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if v > mat_s.max() * 0.6 else "black")
    plt.colorbar(im, ax=ax, label="|Δφᵢ|")
    ax.set_title("Per-feature |Δφᵢ| heatmap\n(sorted by total sensitivity)")

    # Spearman rho
    ax = axes[1, 1]
    lo = np.array(rho_means) - np.array(rho_stds)
    hi = np.array(rho_means) + np.array(rho_stds)
    ax.plot(eps_vals, rho_means, "D-", color="#0F6E56", lw=2, ms=6,
            label="Spearman ρ mean")
    ax.fill_between(eps_vals, lo, hi, alpha=0.18, color="#0F6E56", label="±1 std")
    ax.axhline(0.9, color="#854F0B", ls="--", lw=1.3, label="ρ = 0.90")
    ax.set_ylim(-0.25, 1.08)
    ax.set_xlabel("ε  (relative weight perturbation)")
    ax.set_ylabel("Spearman ρ  (rank correlation with φ₀)")
    ax.set_title("Feature-rank stability vs ε")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig("arm1_adversarial_soil.png", dpi=150, bbox_inches="tight")
    print("  Saved: arm1_adversarial_soil.png")
    plt.close(fig)
    return eps_star


def _plot_arm2(rows: List[dict]) -> None:
    if not rows:
        print("  Arm 2: no rows to plot."); return

    n_vals   = [r["n"]        for r in rows]
    mad_vals = [r["mad_mean"] for r in rows]
    rho_vals = [r["spearman"] for r in rows]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.suptitle(
        "ARM 2: Estimation Noise (SEM subsample) — Soil SPI Regression\n"
        "(model + graph + test instances fixed; only SEM training size varies)",
        fontsize=11, fontweight="bold"
    )
    axes[0].plot(n_vals, mad_vals, "o-", color="#185FA5", lw=2, ms=7)
    axes[0].set_xlabel("SEM training size  n"); axes[0].set_ylabel("MAD(φ_n − φ_full)")
    axes[0].set_title("Attribution drift vs SEM data size")
    axes[0].grid(alpha=0.3); axes[0].set_xticks(n_vals)

    axes[1].plot(n_vals, rho_vals, "s-", color="#0F6E56", lw=2, ms=7)
    axes[1].axhline(0.9, color="#854F0B", ls="--", lw=1.3, label="ρ = 0.90")
    axes[1].set_ylim(-0.25, 1.08)
    axes[1].set_xlabel("SEM training size  n"); axes[1].set_ylabel("Spearman ρ")
    axes[1].set_title("Feature-rank stability vs SEM data size")
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3); axes[1].set_xticks(n_vals)

    fig.tight_layout()
    fig.savefig("arm2_estimation_soil.png", dpi=150, bbox_inches="tight")
    print("  Saved: arm2_estimation_soil.png")
    plt.close(fig)


def _plot_gamma_sensitivity(arm1_rows: List[dict],
                            gamma: Dict[str, float],
                            features: List[str]) -> None:
    max_row     = max((r for r in arm1_rows if r["epsilon"] > 0.0),
                      key=lambda r: r["epsilon"])
    sensitivity = max_row["per_feature_mad"]
    gamma_vals  = np.array([gamma[f] for f in features])
    corr, _     = spearmanr(gamma_vals, sensitivity)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        "Mechanistic check — Soil SPI Regression\n"
        f"Spearman ρ (γᵢ vs sensitivity at ε=max): {corr:.3f}",
        fontsize=11, fontweight="bold"
    )

    ax = axes[0]
    ax.scatter(gamma_vals, sensitivity, s=110, color="#185FA5", zorder=3)
    for f, gv, sv in zip(features, gamma_vals, sensitivity):
        ax.annotate(f.split()[0], (gv, sv), textcoords="offset points",
                    xytext=(6, 4), fontsize=9)
    if len(features) > 2:
        c  = np.polyfit(gamma_vals, sensitivity, 1)
        xs = np.linspace(gamma_vals.min(), gamma_vals.max(), 100)
        ax.plot(xs, np.polyval(c, xs), "--", color="#A32D2D", lw=1.5, label="OLS trend")
    ax.set_xlabel("γᵢ  (normalised total causal effect)", fontsize=11)
    ax.set_ylabel("|Δφᵢ| at ε=max  (attribution shift)", fontsize=11)
    ax.set_title("γᵢ vs attribution sensitivity"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = axes[1]
    order  = np.argsort(gamma_vals)[::-1]
    feat_s = [features[i] for i in order]
    gam_s  = gamma_vals[order]
    sens_s = sensitivity[order]
    x      = np.arange(len(feat_s)); width = 0.38
    ax.bar(x - width/2, gam_s,  width, color="#185FA5", alpha=0.8, label="γᵢ")
    ax.bar(x + width/2, sens_s / (sens_s.max() + 1e-12), width,
           color="#E08020", alpha=0.8, label="|Δφᵢ| normalised")
    ax.set_xticks(x)
    ax.set_xticklabels([f.split()[0] for f in feat_s], rotation=30, ha="right")
    ax.set_ylabel("Score"); ax.set_title("γᵢ vs sensitivity (normalised)")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig("gamma_vs_sensitivity_soil.png", dpi=150, bbox_inches="tight")
    print("  Saved: gamma_vs_sensitivity_soil.png")
    plt.close(fig)


# 10. main pipeline

def run(cfg: dict = CFG) -> dict:
    print(f"\n{'='*72}")
    print("  CAUSAL SHAP ROBUSTNESS — SOIL SPI REGRESSION")
    print(f"{'='*72}\n")
    print(f"MC settings: n_perms={cfg['n_perms']}  M_per_sem={cfg['M_per_sem']}  "
          f"k={cfg['k_instances']}  antithetic={cfg['use_antithetic']}")
    print(f"Effective permutations per instance: "
          f"{cfg['n_perms'] * (2 if cfg['use_antithetic'] else 1)}\n")

    master_rng = np.random.default_rng(cfg["random_seed"])

    Xtr, Xte, y_train, y_test = load_soil_data(cfg)
    features = FEATURE_NAMES

    gamma = load_gamma(cfg["ground_truth_path"], verbose=True)

    print("Training regression model ...")
    if HAVE_XGB:
        model = XGBRegressor(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=cfg["random_seed"], verbosity=0,
        )
        print("Using XGBRegressor")
    else:
        model = RandomForestRegressor(
            n_estimators=160, max_depth=6, random_state=cfg["random_seed"])
        print("Using RandomForestRegressor")

    model.fit(Xtr.values, y_train.values)
    y_pred = model.predict(Xte.values)
    print(f"RMSE={np.sqrt(mean_squared_error(y_test, y_pred)):.4f}  "
          f"MAE={mean_absolute_error(y_test, y_pred):.4f}  "
          f"R²={r2_score(y_test, y_pred):.4f}\n")

    dag       = build_known_dag()
    sems_full = [SEM(dag, features).fit(Xtr)]
    print("Baseline SEM fitted.")
    print("\nBaseline SEM structural coefficients:")
    for var in features:
        m = sems_full[0].mechs[var]
        if m["parents"]:
            for p, c in zip(m["parents"], m["coef"]):
                print(f"  {p:<28s} -> {var:<28s}  β = {c:+.6f}")
    print()

    Ef = _E_f(model, Xtr)
    print(f"E[f(X)] = {Ef:.6f}")

    k                = min(cfg["k_instances"], len(Xte))
    instance_indices = master_rng.choice(len(Xte), size=k, replace=False)
    perm_seeds       = [int(master_rng.integers(0, 2**32))
                        for _ in range(cfg["n_perms"])]
    noise_seeds      = [int(master_rng.integers(0, 2**32))
                        for _ in range(len(sems_full) * cfg["M_per_sem"] * 30)]

    print(f"\nShared seeds locked:")
    print(f"  {k} instances | {cfg['n_perms']} perm seeds | "
          f"{len(noise_seeds)} noise seeds\n")

    print(f"Computing baseline SHAP "
          f"(n_perms={cfg['n_perms']}, M_per_sem={cfg['M_per_sem']}, k={k}) ...")
    base_mechs   = [sem.mechs for sem in sems_full]
    baseline_phi = _shap_global(
        Xte, features, sems_full, base_mechs, gamma,
        model, Ef,
        instance_indices=instance_indices,
        n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        use_antithetic=cfg["use_antithetic"],
        scalar_guard=cfg["scalar_guard"],
        verbose=True,
    )

    print("\nBaseline consistency check (sum(phi) ≈ f(x) - E[f(X)]):")
    ok = _check_baseline_consistency(
        baseline_phi, Xte, instance_indices, model, Ef, features, tol=0.05
    )
    if ok:
        print("  All instances within 5% tolerance. Proceeding.\n")
    else:
        print("  WARNING: Some instances outside tolerance. "
              "Consider increasing n_perms.\n")

    phi0_mean = baseline_phi.mean(axis=0)
    print("Baseline φ₀ (sorted by |φ|):")
    for f, v in sorted(zip(features, phi0_mean), key=lambda x: -abs(x[1])):
        print(f"  {f:<30}  {v:+.6f}")

    print(f"\n{'─'*60}")
    print("ARM 1: Adversarial epsilon sweep")
    print(f"  ε = {cfg['epsilons']}")
    print(f"  {cfg['n_trials']} trials per ε")
    print(f"{'─'*60}")
    arm1_rows = _arm1_sweep(
        baseline_phi, Xte, features, sems_full, gamma,
        model, Ef, instance_indices,
        perm_seeds, noise_seeds, master_rng, cfg,
    )

    noise_floor = next((r["mad_mean"] for r in arm1_rows if r["epsilon"] == 0.0), 0.0)
    ref = np.abs(phi0_mean).mean()
    print(f"\n  MC noise floor (MAD at ε=0) : {noise_floor:.6f}")
    if ref > 1e-10:
        pct = 100 * noise_floor / ref
        print(f"  Noise floor as % of |φ₀|    : {pct:.2f}%")
        print("  OK: noise floor negligible." if pct <= 5.0
              else "  WARNING: noise floor > 5% — increase n_perms / M_per_sem.")
    _check_monotonicity(arm1_rows)

    eps_star = _plot_arm1(arm1_rows, features, phi0_mean, cfg)
    print(f"  ε* = {eps_star}  "
          f"({'breakdown reached' if eps_star else 'stable across all ε'})")

    records = []
    for r in arm1_rows:
        rec = {
            "epsilon":       r["epsilon"],
            "mad_mean":      r["mad_mean"],
            "mad_std":       r["mad_std"],
            "mad_corrected": max(0.0, r["mad_mean"] - noise_floor),
            "spearman_mean": r["spearman_mean"],
            "spearman_std":  r["spearman_std"],
        }
        for i, f in enumerate(features):
            rec[f"mad_{f}"] = r["per_feature_mad"][i]
        records.append(rec)
    arm1_df = pd.DataFrame(records).set_index("epsilon")
    arm1_df.to_csv("arm1_scores_soil.csv")
    print("  Saved: arm1_scores_soil.csv")
    print(arm1_df[["mad_mean", "mad_corrected", "spearman_mean"]].round(6).to_string())

    print(f"\n{'─'*60}")
    print("ARM 2: Estimation-noise sweep")
    print(f"  n = {cfg['subsample_sizes']}")
    print(f"{'─'*60}")
    arm2_rows = _arm2_sweep(
        baseline_phi, Xtr, Xte, features, dag,
        gamma, model, Ef, instance_indices,
        perm_seeds, noise_seeds, master_rng, cfg,
    )
    _plot_arm2(arm2_rows)

    records2 = []
    for r in arm2_rows:
        rec = {"n": r["n"], "mad": r["mad_mean"], "spearman": r["spearman"]}
        for i, f in enumerate(features):
            rec[f"mad_{f}"] = r["per_feature_mad"][i]
        records2.append(rec)
    arm2_df = pd.DataFrame(records2).set_index("n")
    arm2_df.to_csv("arm2_scores_soil.csv")
    print("  Saved: arm2_scores_soil.csv")
    print(arm2_df[["mad", "spearman"]].round(6).to_string())

    _plot_gamma_sensitivity(arm1_rows, gamma, features)

    print(f"\n{'='*72}")
    print("  ROBUSTNESS ANALYSIS COMPLETE — Soil SPI Regression")
    print(f"{'='*72}")
    idx = int(np.argmax(np.abs(phi0_mean)))
    print(f"  Dominant feature : {features[idx]}  φ₀ = {phi0_mean[idx]:+.4f}")
    print(f"  MC noise floor   : {noise_floor:.6f}")
    if eps_star:
        print(f"  ε*               : {eps_star:.2f}  "
              f"(weights must be within {eps_star*100:.0f}% relative error)")
    else:
        print("  ε*               : not reached — stable across all ε")

    return {
        "features":    features,
        "phi0_mean":   phi0_mean,
        "gamma":       gamma,
        "noise_floor": noise_floor,
        "eps_star":    eps_star,
        "arm1_rows":   arm1_rows,
        "arm2_rows":   arm2_rows,
    }


if __name__ == "__main__":
    run(CFG)


  CAUSAL SHAP ROBUSTNESS — SOIL SPI REGRESSION

MC settings: n_perms=200  M_per_sem=32  k=10  antithetic=True
Effective permutations per instance: 400

Loaded 5000 rows. SPI range [71.90, 302.20]  mean=163.24  std=32.66
Train: 4000   Test: 1000

gamma_i from ground_truth.csv:
  Organic Carbon                  gamma = 0.062171
  Nitrogen                        gamma = 0.401157
  Phosphorus                      gamma = 0.076187
  Potassium                       gamma = 0.258773
  pH                              gamma = 0.088614
  Zinc                            gamma = 0.014698
  Electrical Conductivity         gamma = 0.098400
  sum                             1.000000

Training regression model ...
Using XGBRegressor
RMSE=3.4633  MAE=2.6991  R²=0.9882

Baseline SEM fitted.

Baseline SEM structural coefficients:
  Organic Carbon               -> pH                            β = -0.708307
  pH                           -> Zinc                          β = -0.194760

E[f(X)] = 163.21887